import all libraries

In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

connect to db

In [43]:
load_dotenv()
db_url = os.getenv("DATABASE_URL")
engine = create_engine(db_url)
player = pd.read_sql("SELECT * FROM playerstat", engine)
league = pd.read_sql("SELECT * FROM leaguestat", engine)
league.head()

,Season,PA,RS,IP,G,1B,BB,IBB,HBP,SB,CS,GDP,GDP_OPP,league_wOBA
0,1871,11215.0,2659,2250.000000,277,2381,393,0.0,0.0,441.0,123.0,74.0,10775.0,0.312350
1,1872,15928.0,3390,3286.000000,405,3704,263,0.0,0.0,269.0,134.0,97.0,15628.0,0.285315
2,1873,17294.0,3580,3584.666667,434,4098,335,0.0,0.0,314.0,131.0,122.0,16912.0,0.293449
3,1874,19342.0,3470,4169.666667,490,4356,238,0.0,0.0,242.0,97.0,107.0,19064.0,0.270910
4,1875,27082.0,4234,6190.333333,763,5660,249,0.0,0.0,629.0,320.0,142.0,26793.0,0.250098


Combine player dataframe and league dataframe

In [44]:
league = league.rename(columns={"Season": "season"})
combined = pd.merge(player, league, on="season")
pd.set_option('display.max_columns', None)
combined.head()

,idfg,name,season,age,pa,ab,h,1B_x,2B,3B,hr,bb,ibb,ubb,hbp,sb,cs,gdp,gdp_opp,sf,woba,PA,RS,IP,G,1B_y,BB,IBB,HBP,SB,CS,GDP,GDP_OPP,league_wOBA
0,aardsda01,David Aardsma,2004,23.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,188519.0,23376,43394.000000,18272,29254,16222,1381.0,1850.0,2589.0,1100.0,3784.0,163633.0,0.333745
1,aardsda01,David Aardsma,2006,25.0,3.0,2,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,188052.0,23599,43258.000000,18694,29600,15847,1410.0,1817.0,2767.0,1110.0,3945.0,163606.0,0.335818
2,aardsda01,David Aardsma,2007,26.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,188598.0,23322,43425.666667,19294,29885,16079,1323.0,1755.0,2918.0,1002.0,3983.0,164366.0,0.332051
3,aardsda01,David Aardsma,2008,27.0,1.0,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,187614.0,22585,43357.666667,19012,29194,16337,1310.0,1672.0,2799.0,1035.0,3883.0,163362.0,0.328572
4,aardsda01,David Aardsma,2009,28.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,187060.0,22419,43272.000000,19097,28796,16620,1179.0,1590.0,2970.0,1133.0,3796.0,162442.0,0.328832


Compute wRAA, BsR, RLR(https://www.youtube.com/watch?v=hlFIipW2Gfs)

In [50]:
Sabermetric = pd.DataFrame()
Sabermetric['IDfg'] = combined['idfg']
Sabermetric['Age'] = combined['age']
Sabermetric['Season'] = combined['season']
Sabermetric['wRAA'] = ((combined['woba'] - combined['league_wOBA'])/1.25) * combined['pa'] #Used wOBA Scale = 1.25 

Sabermetric['wGDP'] = (((combined['GDP']/combined['GDP_OPP'])*combined['gdp_opp'])-combined['gdp'])*(combined['RS']/(3*combined['IP']))
Sabermetric['runCS'] = (-2*(combined['RS']/(combined['IP']*3))) + 0.075
Sabermetric['league_wSB'] = ((combined['SB']*0.2) + (combined['CS']*Sabermetric['runCS']))/(combined['1B_y']+combined['BB']+combined['HBP']+combined['IBB'])
Sabermetric['wSB'] = ((combined['sb']*0.2) + (combined['cs']*Sabermetric['runCS']))-(Sabermetric['league_wSB']*combined['1B_y']+combined['BB']+combined['HBP']+combined['IBB'])
Sabermetric['BsR'] = Sabermetric['wSB'] + Sabermetric['wGDP']

Sabermetric['RPW'] = (9*(combined['RS']/(combined['IP']*3))*1.5) + 3 #assume 9 innings per game
Sabermetric['RLR'] = ((0.235*2430)  * Sabermetric['RPW'] * combined['pa'])/combined['PA'] #assume full season, 162 games for 30 teams, divide by 2 to count wins and losses

Sabermetric = Sabermetric[['IDfg', 'Age', 'Season', 'wRAA', 'BsR', 'RLR']]
Sabermetric = Sabermetric.dropna()

Start ML